# ERA5 local-solar heat climatology and hazard targets

This notebook builds a reproducible ERA5 heat-hazard dataset from six-hourly 2 m temperature.

It produces, for **daily mean**, **sampled daily minimum**, and **sampled daily maximum**:

1. Local-solar daily temperature fields.
2. Calendar-day percentile thresholds for 1991–2020.
3. q95 exceedance masks.
4. Onset masks for heat events lasting at least 2 and 3 consecutive days.
5. Validation summaries and diagnostic figures.

All outputs are written below:

```text
/net/monsoon/kylehall/ERA5/heat_extremes_climatology/
├── daily/
├── thresholds/
├── hazards/
├── diagnostics/
└── figures/
```

The local-solar day approximation uses longitude bands with UTC offsets quantized to six-hour increments. This is appropriate for the six-hourly source data but does not recover exact civil-day extrema.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import cartopy.crs as ccrs

from dask.diagnostics import ProgressBar
from xclim.core.calendar import percentile_doy, resample_doy

import heatextremes as he

xr.set_options(keep_attrs=True)


## Configuration

In [2]:
# Source period
START_YEAR = 1991
END_YEAR = 2026

# Climatological reference period
CLIM_START_YEAR = 1991
CLIM_END_YEAR = 2020

# Daily statistics derived from six-hourly samples
STATISTICS = ("mean", "min", "max")

# Calendar-day percentile configuration
PERCENTILES = (90.0, 95.0, 97.5)
PRIMARY_PERCENTILE = 95.0
WINDOW_DAYS = 15

# Event-duration definitions
EVENT_DURATIONS = (2, 3)

# Output layout
OUTPUT_ROOT = Path("/net/monsoon/kylehall/ERA5/heat_extremes_climatology")
DAILY_DIR = OUTPUT_ROOT / "daily"
THRESHOLD_DIR = OUTPUT_ROOT / "thresholds"
HAZARD_DIR = OUTPUT_ROOT / "hazards"
DIAGNOSTIC_DIR = OUTPUT_ROOT / "diagnostics"
FIGURE_DIR = OUTPUT_ROOT / "figures"

for directory in (
    OUTPUT_ROOT,
    DAILY_DIR,
    THRESHOLD_DIR,
    HAZARD_DIR,
    DIAGNOSTIC_DIR,
    FIGURE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

SOURCE_CHUNKS = {
    "time": 2048,
    "latitude": 180,
    "longitude": 360,
}

DAILY_OUTPUT_CHUNKS = {
    "time": 360,
    "latitude": 180,
    "longitude": 180,
}

PERCENTILE_SPATIAL_CHUNKS = {
    "latitude": 45,
    "longitude": 90,
}

print(f"Outputs will be written to: {OUTPUT_ROOT}")


Outputs will be written to: /net/monsoon/kylehall/ERA5/heat_extremes_climatology


## Utility functions

In [3]:
def clear_chunk_encoding(obj: xr.Dataset | xr.DataArray) -> xr.Dataset | xr.DataArray:
    """Remove inherited Zarr chunk metadata that can conflict with new chunks."""
    obj = obj.copy()
    variables = obj.variables if isinstance(obj, xr.Dataset) else {obj.name or "__data__": obj}

    if isinstance(obj, xr.Dataset):
        for name in obj.variables:
            obj[name].encoding.pop("chunks", None)
            obj[name].encoding.pop("preferred_chunks", None)
    else:
        obj.encoding.pop("chunks", None)
        obj.encoding.pop("preferred_chunks", None)

    return obj


def write_zarr(
    obj: xr.Dataset | xr.DataArray,
    path: Path,
    *,
    mode: str = "w",
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    if isinstance(obj, xr.DataArray):
        if obj.name is None:
            raise ValueError("DataArray must have a name before writing.")
        obj = obj.to_dataset()

    obj = clear_chunk_encoding(obj)

    with ProgressBar():
        obj.to_zarr(
            path,
            mode=mode,
            consolidated=True,
            zarr_format=2,
        )


def area_weighted_global_mean(da: xr.DataArray) -> xr.DataArray:
    """Cosine-latitude-weighted global mean."""
    weights = np.cos(np.deg2rad(da["latitude"]))
    return da.weighted(weights).mean(("latitude", "longitude"))


In [4]:
def aggregate_six_hourly_band(
    band: xr.DataArray,
    offset_hours: int,
    *,
    statistic: str,
    time_dim: str = "time",
) -> xr.DataArray:
    """
    Aggregate regular six-hourly UTC data into approximate local-solar days.

    The longitude-band UTC offset must be a multiple of six hours.
    Incomplete edge days are discarded.
    """
    if offset_hours % 6 != 0:
        raise ValueError("offset_hours must be a multiple of six.")

    if statistic not in {"mean", "min", "max"}:
        raise ValueError("statistic must be 'mean', 'min', or 'max'.")

    shifted_time = band[time_dim] + pd.to_timedelta(offset_hours, unit="h")
    midnight_indices = np.flatnonzero(shifted_time.dt.hour.values == 0)

    if midnight_indices.size == 0:
        raise ValueError(
            "No local-midnight sample was found. Verify regular six-hourly input."
        )

    first = int(midnight_indices[0])
    aligned = band.isel({time_dim: slice(first, None)})
    shifted_time = shifted_time.isel({time_dim: slice(first, None)})

    n_samples = (aligned.sizes[time_dim] // 4) * 4
    if n_samples == 0:
        raise ValueError("No complete local-solar day is available.")

    aligned = aligned.isel({time_dim: slice(0, n_samples)})
    shifted_time = shifted_time.isel({time_dim: slice(0, n_samples)})

    grouped = aligned.coarsen(
        {time_dim: 4},
        boundary="exact",
        coord_func={time_dim: "min"},
    )
    daily = getattr(grouped, statistic)()

    local_dates = (
        shifted_time
        .isel({time_dim: slice(0, None, 4)})
        .dt.floor("D")
    )
    daily = daily.assign_coords({time_dim: local_dates})

    daily.attrs = band.attrs.copy()
    daily.attrs.update(
        daily_statistic=statistic,
        local_solar_offset_hours=offset_hours,
    )
    return daily


def local_solar_daily_stat(
    da: xr.DataArray,
    *,
    statistic: str,
    time_dim: str = "time",
    lon_dim: str = "longitude",
) -> xr.DataArray:
    """
    Compute an approximate local-solar daily statistic from six-hourly data.

    Longitude-dependent UTC offsets are quantized to six-hour increments.
    The returned time coordinate contains dates common to all longitude bands.
    """
    lon_values = np.asarray(da[lon_dim].values)

    if np.any(np.diff(lon_values) <= 0):
        raise ValueError("Longitude must be strictly increasing.")

    uses_360 = lon_values.min() >= 0 and lon_values.max() > 180
    uses_180 = lon_values.min() < 0 and lon_values.max() <= 180

    if not (uses_360 or uses_180):
        raise ValueError("Expected longitude in [0, 360) or [-180, 180).")

    def select_half_open(
        array: xr.DataArray,
        start: float,
        stop: float,
    ) -> xr.DataArray:
        return array.sel(
            {lon_dim: slice(start, np.nextafter(stop, -np.inf))}
        )

    if uses_180:
        band_specs = (
            (-180.0, -135.0, -12),
            (-135.0,  -45.0,  -6),
            ( -45.0,   45.0,   0),
            (  45.0,  135.0,   6),
            ( 135.0,  180.0,  12),
        )
    else:
        band_specs = (
            (  0.0,   45.0,   0),
            ( 45.0,  135.0,   6),
            (135.0,  180.0,  12),
            (180.0,  225.0, -12),
            (225.0,  315.0,  -6),
            (315.0,  360.0,   0),
        )

    pieces = []
    for start, stop, offset in band_specs:
        band = select_half_open(da, start, stop)
        if band.sizes.get(lon_dim, 0) == 0:
            continue

        daily = aggregate_six_hourly_band(
            band,
            offset,
            statistic=statistic,
            time_dim=time_dim,
        )
        daily = daily.assign_coords(
            local_solar_offset_hours=xr.full_like(
                daily[lon_dim],
                offset,
                dtype=np.int8,
            )
        )
        pieces.append(daily)

    result = xr.concat(
        pieces,
        dim=lon_dim,
        join="inner",
        coords="minimal",
        compat="override",
    )

    if not np.all(np.diff(result[lon_dim].values) > 0):
        raise RuntimeError("Longitude bands were concatenated out of order.")

    result.attrs.update(
        daily_time_basis=(
            "Approximate local-solar day using six-hour UTC-offset bands."
        ),
        source_sampling="six-hourly",
    )
    return result


In [5]:
def calculate_calendar_day_percentiles(
    daily: xr.DataArray,
    *,
    start_year: int,
    end_year: int,
    percentiles: Iterable[float],
    window: int,
    time_dim: str = "time",
    spatial_chunks: dict[str, int] | None = None,
) -> xr.DataArray:
    """Calculate moving-window calendar-day percentile thresholds."""
    if window < 1 or window % 2 == 0:
        raise ValueError("window must be a positive odd integer.")

    climatology = daily.sel(
        {time_dim: slice(f"{start_year}-01-01", f"{end_year}-12-31")}
    )

    expected_years = np.arange(start_year, end_year + 1)
    available_years = np.unique(climatology[time_dim].dt.year.values)
    missing_years = np.setdiff1d(expected_years, available_years)

    if missing_years.size:
        raise ValueError(f"Missing climatology years: {missing_years.tolist()}")

    chunks = {time_dim: -1}
    if spatial_chunks:
        chunks.update(spatial_chunks)
    chunks = {dim: size for dim, size in chunks.items() if dim in climatology.dims}
    climatology = climatology.chunk(chunks)

    thresholds = percentile_doy(
        climatology,
        window=window,
        per=list(percentiles),
    )

    thresholds.name = f"{daily.name}_calendar_day_percentile"
    thresholds.attrs.update(
        climatology_period=f"{start_year}-{end_year}",
        climatology_window_days=window,
        climatology_window_half_width_days=window // 2,
        threshold_type="calendar-day percentile",
        daily_time_basis=daily.attrs.get("daily_time_basis", "unspecified"),
    )
    return thresholds


def heatwave_start_mask(
    hot: xr.DataArray,
    min_duration: int,
    *,
    time_dim: str = "time",
) -> xr.DataArray:
    """Mark only the first day of each run of at least min_duration hot days."""
    if min_duration < 1:
        raise ValueError("min_duration must be at least one.")

    hot = hot.fillna(False).astype(bool)

    qualifies = xr.concat(
        [
            hot.shift({time_dim: -lag}, fill_value=False)
            for lag in range(min_duration)
        ],
        dim="_duration_check",
    ).all("_duration_check")

    previous_hot = hot.shift({time_dim: 1}, fill_value=False)
    return (qualifies & ~previous_hot).rename(
        f"heatwave_start_{min_duration}d"
    )


def validate_heatwave_start_mask(
    hot: xr.DataArray,
    starts: xr.DataArray,
    min_duration: int,
    *,
    time_dim: str = "time",
) -> None:
    """Assert the defining invariants for a computed onset mask."""
    expected = hot.fillna(False).astype(bool)
    for lag in range(1, min_duration):
        expected = expected & hot.shift(
            {time_dim: -lag},
            fill_value=False,
        ).astype(bool)

    expected = expected & ~hot.shift(
        {time_dim: 1},
        fill_value=False,
    ).astype(bool)

    np.testing.assert_array_equal(
        starts.astype(bool).values,
        expected.values,
    )


## Open source ERA5

In [6]:
era5 = he.open_cached_era5(
    start_year=START_YEAR,
    chunks=SOURCE_CHUNKS,
)

t2m = era5["2m_temperature"]

print(t2m)
print("Time range:", str(t2m.time.values[0]), "to", str(t2m.time.values[-1]))


<xarray.DataArray '2m_temperature' (time: 51928, latitude: 721, longitude: 1440)> Size: 216GB
dask.array<concatenate, shape=(51928, 721, 1440), dtype=float32, chunksize=(1464, 180, 360), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 415kB 1991-01-01 ... 2026-07-17T18:00:00
  * latitude   (latitude) float64 6kB -90.0 -89.75 -89.5 ... 89.5 89.75 90.0
  * longitude  (longitude) float64 12kB -180.0 -179.8 -179.5 ... 179.5 179.8
Attributes: (12/30)
    GRIB_NV:                                  0
    GRIB_Nx:                                  1440
    GRIB_Ny:                                  721
    GRIB_cfName:                              unknown
    GRIB_cfVarName:                           t2m
    GRIB_dataType:                            an
    ...                                       ...
    GRIB_totalNumber:                         0
    GRIB_typeOfLevel:                         surface
    GRIB_units:                               K
    long_name:       

## 1. Build local-solar daily temperature fields

This cell writes one store per statistic:

```text
daily/t2m_daily_mean.zarr
daily/t2m_daily_min.zarr
daily/t2m_daily_max.zarr
```

In [7]:
for statistic in STATISTICS:
    output_path = DAILY_DIR / f"t2m_daily_{statistic}.zarr"

    print(f"\nBuilding {statistic}: {output_path}")
    daily = local_solar_daily_stat(t2m, statistic=statistic)
    daily = daily.rename(f"t2m_daily_{statistic}")
    daily = daily.chunk(DAILY_OUTPUT_CHUNKS)
    daily.attrs.update(
        statistic=statistic,
        source_variable="ERA5 2m_temperature",
    )

    # The longitude-only offset coordinate is tiny and safer to write eagerly.
    if "local_solar_offset_hours" in daily.coords:
        daily = daily.assign_coords(
            local_solar_offset_hours=(
                "longitude",
                daily["local_solar_offset_hours"].values,
            )
        )

    write_zarr(daily, output_path)



Building mean: /net/monsoon/kylehall/ERA5/heat_extremes_climatology/daily/t2m_daily_mean.zarr
[########################################] | 100% Completed | 205.59 s

Building min: /net/monsoon/kylehall/ERA5/heat_extremes_climatology/daily/t2m_daily_min.zarr
[########################################] | 100% Completed | 151.62 s

Building max: /net/monsoon/kylehall/ERA5/heat_extremes_climatology/daily/t2m_daily_max.zarr
[########################################] | 100% Completed | 150.71 s


## 2. Build 1991–2020 calendar-day percentile thresholds

Each store contains q90, q95, and q97.5 with a centered 15-day moving window.

In [8]:
for statistic in STATISTICS:
    daily_path = DAILY_DIR / f"t2m_daily_{statistic}.zarr"
    output_path = (
        THRESHOLD_DIR
        / f"t2m_daily_{statistic}_percentiles_"
          f"{CLIM_START_YEAR}_{CLIM_END_YEAR}.zarr"
    )

    print(f"\nBuilding thresholds for {statistic}: {output_path}")

    daily = xr.open_zarr(
        daily_path,
        consolidated=True,
        chunks={},
    )[f"t2m_daily_{statistic}"]

    # Entire climatology time series must be available within each
    # spatial block for percentile_doy.
    daily = daily.chunk(
        {
            "time": -1,
            "latitude": 45,
            "longitude": 90,
        }
    )

    thresholds = calculate_calendar_day_percentiles(
        daily,
        start_year=CLIM_START_YEAR,
        end_year=CLIM_END_YEAR,
        percentiles=PERCENTILES,
        window=WINDOW_DAYS,
        spatial_chunks=None,
    )

    # percentile_doy creates irregular spatial chunks.
    # Rechunk explicitly to a uniform Zarr-compatible layout.
    thresholds = thresholds.chunk(
        {
            "latitude": 45,
            "longitude": 90,
            "dayofyear": 366,
            "percentiles": len(PERCENTILES),
        }
    )

    thresholds.attrs.update(
        climatology_period=f"{CLIM_START_YEAR}-{CLIM_END_YEAR}",
        climatology_window_days=WINDOW_DAYS,
        climatology_window_half_width_days=WINDOW_DAYS // 2,
        threshold_type="calendar-day percentile",
    )

    if "local_solar_offset_hours" in thresholds.coords:
        thresholds = thresholds.assign_coords(
            local_solar_offset_hours=(
                "longitude",
                thresholds["local_solar_offset_hours"].values,
            )
        )

    print("Output chunks:", thresholds.chunks)
    print("Dask tasks:", len(thresholds.data.__dask_graph__()))
    print("Dask layers:", len(thresholds.data.dask.layers))

    write_zarr(thresholds, output_path)


Building thresholds for mean: /net/monsoon/kylehall/ERA5/heat_extremes_climatology/thresholds/t2m_daily_mean_percentiles_1991_2020.zarr


/home/kylehall/miniconda3/envs/heat-extremes/lib/python3.11/site-packages/dask/array/core.py:5201: PerformanceWarning: Increasing number of chunks by factor of 64
  result = blockwise(


Output chunks: ((45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 1), (90, 90, 90, 90, 90, 90, 90, 90, 90, 90, 90, 90, 90, 90, 90, 90), (366,), (3,))
Dask tasks: 851746
Dask layers: 52
[########################################] | 100% Completed | 13m 18s

Building thresholds for min: /net/monsoon/kylehall/ERA5/heat_extremes_climatology/thresholds/t2m_daily_min_percentiles_1991_2020.zarr


/home/kylehall/miniconda3/envs/heat-extremes/lib/python3.11/site-packages/dask/array/core.py:5201: PerformanceWarning: Increasing number of chunks by factor of 64
  result = blockwise(


Output chunks: ((45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 1), (90, 90, 90, 90, 90, 90, 90, 90, 90, 90, 90, 90, 90, 90, 90, 90), (366,), (3,))
Dask tasks: 851746
Dask layers: 52
[########################################] | 100% Completed | 13m 13s

Building thresholds for max: /net/monsoon/kylehall/ERA5/heat_extremes_climatology/thresholds/t2m_daily_max_percentiles_1991_2020.zarr


/home/kylehall/miniconda3/envs/heat-extremes/lib/python3.11/site-packages/dask/array/core.py:5201: PerformanceWarning: Increasing number of chunks by factor of 64
  result = blockwise(


Output chunks: ((45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 1), (90, 90, 90, 90, 90, 90, 90, 90, 90, 90, 90, 90, 90, 90, 90, 90), (366,), (3,))
Dask tasks: 851746
Dask layers: 52
[########################################] | 100% Completed | 13m 34s


## 3. Build q95 exceedance and heat-event onset targets

For each daily statistic, this creates:

- `hot_day_q95`
- `heatwave_start_q95_2d`
- `heatwave_start_q95_3d`

The terminology is generic in the files: for daily minimum, an exceedance is most naturally interpreted as a **hot night**; for daily maximum, as a **hot daytime extreme**.

In [10]:
hazard_paths = {}

for statistic in STATISTICS:
    daily_path = DAILY_DIR / f"t2m_daily_{statistic}.zarr"
    threshold_path = (
        THRESHOLD_DIR
        / f"t2m_daily_{statistic}_percentiles_{CLIM_START_YEAR}_{CLIM_END_YEAR}.zarr"
    )
    output_path = HAZARD_DIR / f"t2m_daily_{statistic}_q95_hazards.zarr"

    print(f"\nBuilding q95 hazards for {statistic}: {output_path}")

    daily = xr.open_zarr(daily_path, consolidated=True, chunks={})[
        f"t2m_daily_{statistic}"
    ]
    thresholds = xr.open_zarr(
        threshold_path,
        consolidated=True,
        chunks={},
    )[f"t2m_daily_{statistic}_calendar_day_percentile"]

    q95 = thresholds.sel(percentiles=PRIMARY_PERCENTILE, drop=True)
    q95_for_dates = resample_doy(q95, daily)

    hot = (daily > q95_for_dates).rename("hot_day_q95")
    hot.attrs = {
        "long_name": (
            f"Daily {statistic} 2 m temperature exceeds the local-solar "
            f"{CLIM_START_YEAR}-{CLIM_END_YEAR} calendar-day q95 threshold"
        ),
        "units": "1",
        "daily_statistic": statistic,
        "percentile": PRIMARY_PERCENTILE,
        "climatology_period": f"{CLIM_START_YEAR}-{CLIM_END_YEAR}",
        "climatology_window_days": WINDOW_DAYS,
    }

    hazard_variables = {"hot_day_q95": hot}
    for duration in EVENT_DURATIONS:
        onset = heatwave_start_mask(hot, min_duration=duration)
        name = f"heatwave_start_q95_{duration}d"
        onset = onset.rename(name)
        onset.attrs = {
            "long_name": (
                f"First day of a run of at least {duration} consecutive days "
                f"whose daily {statistic} 2 m temperature exceeds q95"
            ),
            "units": "1",
            "minimum_duration_days": duration,
            "daily_statistic": statistic,
            "percentile": PRIMARY_PERCENTILE,
        }
        hazard_variables[name] = onset

    hazards = xr.Dataset(hazard_variables).chunk(DAILY_OUTPUT_CHUNKS)
    hazards.attrs.update(
        title=f"ERA5 daily-{statistic} q95 heat-hazard targets",
        source="ERA5 six-hourly 2 m temperature",
        daily_time_basis="Approximate local-solar day using six-hour offset bands",
        climatology_period=f"{CLIM_START_YEAR}-{CLIM_END_YEAR}",
    )

    if "local_solar_offset_hours" in hazards.coords:
        hazards = hazards.assign_coords(
            local_solar_offset_hours=(
                "longitude",
                hazards["local_solar_offset_hours"].values,
            )
        )

    write_zarr(hazards, output_path)
    hazard_paths[statistic] = output_path



Building q95 hazards for mean: /net/monsoon/kylehall/ERA5/heat_extremes_climatology/hazards/t2m_daily_mean_q95_hazards.zarr


KeyboardInterrupt: 

## 4. Programmatic validation

In [ ]:
# A single point is sufficient to validate the temporal logic.
validation_point = {
    "latitude": 41.88,
    "longitude": 360.0 - 87.63,
}

validation_rows = []

for statistic, hazard_path in hazard_paths.items():
    hazards = xr.open_zarr(
        hazard_path,
        consolidated=True,
        chunks={},
    )

    hot = hazards["hot_day_q95"].sel(
        **validation_point,
        method="nearest",
    ).compute()

    counts = {}
    for duration in EVENT_DURATIONS:
        name = f"heatwave_start_q95_{duration}d"
        starts = hazards[name].sel(
            **validation_point,
            method="nearest",
        ).compute()

        validate_heatwave_start_mask(
            hot,
            starts,
            min_duration=duration,
        )
        counts[duration] = int(starts.sum().item())

    # A stricter duration cannot yield more event starts.
    ordered = sorted(EVENT_DURATIONS)
    for shorter, longer in zip(ordered[:-1], ordered[1:]):
        assert counts[shorter] >= counts[longer]

    validation_rows.append(
        {
            "statistic": statistic,
            "latitude": float(hot.latitude),
            "longitude": float(hot.longitude),
            "hot_days": int(hot.sum().item()),
            **{
                f"events_{duration}d": counts[duration]
                for duration in EVENT_DURATIONS
            },
        }
    )

validation = pd.DataFrame(validation_rows)
validation_path = DIAGNOSTIC_DIR / "event_detector_validation.csv"
validation.to_csv(validation_path, index=False)

print("All event-detector invariant checks passed.")
display(validation)


## 5. Climatological frequency diagnostics

In [ ]:
frequency_rows = []
frequency_maps = {}

for statistic, hazard_path in hazard_paths.items():
    hot = xr.open_zarr(
        hazard_path,
        consolidated=True,
        chunks={},
    )["hot_day_q95"]

    base = hot.sel(
        time=slice(
            f"{CLIM_START_YEAR}-01-01",
            f"{CLIM_END_YEAR}-12-31",
        )
    ).mean("time")

    recent = hot.sel(
        time=slice(f"{CLIM_END_YEAR + 1}-01-01", None)
    ).mean("time")

    base_global = area_weighted_global_mean(base).compute().item()
    recent_global = area_weighted_global_mean(recent).compute().item()

    frequency_rows.append(
        {
            "statistic": statistic,
            "base_period": f"{CLIM_START_YEAR}-{CLIM_END_YEAR}",
            "base_global_frequency": base_global,
            "recent_period_start": CLIM_END_YEAR + 1,
            "recent_global_frequency": recent_global,
        }
    )
    frequency_maps[statistic] = {
        "base": base.compute(),
        "recent": recent.compute(),
    }

frequency_summary = pd.DataFrame(frequency_rows)
frequency_summary.to_csv(
    DIAGNOSTIC_DIR / "q95_global_frequency_summary.csv",
    index=False,
)

display(frequency_summary)


In [ ]:
for statistic, maps in frequency_maps.items():
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14, 4.5),
        subplot_kw={"projection": ccrs.PlateCarree()},
        constrained_layout=True,
    )

    for ax, period_name, field in (
        (
            axes[0],
            f"{CLIM_START_YEAR}–{CLIM_END_YEAR}",
            maps["base"],
        ),
        (
            axes[1],
            f"{CLIM_END_YEAR + 1}–present",
            maps["recent"],
        ),
    ):
        field.plot(
            ax=ax,
            transform=ccrs.PlateCarree(),
            vmin=0.0,
            vmax=0.25,
            cbar_kwargs={"label": "q95 exceedance frequency"},
        )
        ax.coastlines()
        ax.set_title(period_name)

    fig.suptitle(f"Daily {statistic} T2M q95 exceedance frequency")
    figure_path = FIGURE_DIR / f"q95_frequency_daily_{statistic}.png"
    fig.savefig(figure_path, dpi=180, bbox_inches="tight")
    plt.show()


## 6. Save run metadata

In [ ]:
run_metadata = {
    "source_start_year": START_YEAR,
    "source_end_year_requested": END_YEAR,
    "climatology_start_year": CLIM_START_YEAR,
    "climatology_end_year": CLIM_END_YEAR,
    "statistics": list(STATISTICS),
    "percentiles": list(PERCENTILES),
    "primary_percentile": PRIMARY_PERCENTILE,
    "window_days": WINDOW_DAYS,
    "event_durations_days": list(EVENT_DURATIONS),
    "daily_output_chunks": DAILY_OUTPUT_CHUNKS,
    "output_root": str(OUTPUT_ROOT),
}

metadata_path = OUTPUT_ROOT / "run_metadata.json"
metadata_path.write_text(
    json.dumps(run_metadata, indent=2),
    encoding="utf-8",
)

print(json.dumps(run_metadata, indent=2))
print(f"\nPipeline complete. Outputs: {OUTPUT_ROOT}")
